# Notebook 02: Huấn Luyện Mô Hình Phân Loại Lâm Sàng XGBoost
### Dự Án: Multimodal Medical AI Chatbot (Phiên Bản 3.0.0)
- **Dataset**: `itachi9604/disease-symptom-description` & `tboyle10/medicaltranscriptions`
- **Thuật toán**: `XGBoost Multi-Class Classifier` trên GPU CUDA
- **Output**: `tabular_xgboost.json` và `label_encoder.pkl`

In [ ]:
# 1. Cài đặt các thư viện
!pip install -q xgboost scikit-learn pandas numpy joblib

In [ ]:
# 2. Import thư viện & Chuẩn bị dữ liệu tổng hợp kết hợp Lab + Triệu chứng
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import joblib

FEATURE_COLS = [
    'WBC', 'PLT', 'RBC', 'HGB', 'HCT', 'AST', 'ALT', 'GLUCOSE', 'CREATININE',
    'has_sot', 'has_dau_dau', 'has_dau_nguc', 'has_kho_tho', 'has_dau_bung',
    'has_non_oi', 'has_tieu_chay', 'has_chay_mau', 'has_meo_mieng'
]

# Sinh dataset mô phỏng 1000 bệnh nhân theo chuẩn y khoa
np.random.seed(42)
N = 1000
data = []
labels = []

for _ in range(N):
    disease_choice = np.random.choice([
        'Dengue_A90', 'Infarction_I21', 'Stroke_I64', 'Pneumonia_J18',
        'CommonCold_J00', 'Gastritis_K29', 'Diabetes_E11', 'LiverDisease_K76'
    ])
    # Default baseline normal values
    row = {
        'WBC': np.random.uniform(4.5, 9.5),
        'PLT': np.random.uniform(180, 350),
        'RBC': np.random.uniform(4.0, 5.2),
        'HGB': np.random.uniform(130, 155),
        'HCT': np.random.uniform(38, 46),
        'AST': np.random.uniform(15, 35),
        'ALT': np.random.uniform(15, 35),
        'GLUCOSE': np.random.uniform(4.2, 5.8),
        'CREATININE': np.random.uniform(60, 95),
        'has_sot': 0, 'has_dau_dau': 0, 'has_dau_nguc': 0, 'has_kho_tho': 0,
        'has_dau_bung': 0, 'has_non_oi': 0, 'has_tieu_chay': 0, 'has_chay_mau': 0, 'has_meo_mieng': 0
    }
    if disease_choice == 'Dengue_A90':
        row['WBC'] = np.random.uniform(2.0, 3.8)
        row['PLT'] = np.random.uniform(25, 95)
        row['has_sot'] = 1
        row['has_dau_dau'] = 1
        row['has_chay_mau'] = np.random.choice([0, 1], p=[0.3, 0.7])
    elif disease_choice == 'Infarction_I21':
        row['has_dau_nguc'] = 1
        row['has_kho_tho'] = 1
    elif disease_choice == 'Stroke_I64':
        row['has_meo_mieng'] = 1
    elif disease_choice == 'Pneumonia_J18':
        row['WBC'] = np.random.uniform(12.5, 22.0)
        row['has_sot'] = 1
        row['has_kho_tho'] = 1
    elif disease_choice == 'CommonCold_J00':
        row['has_sot'] = 1
        row['has_dau_dau'] = 1
    elif disease_choice == 'Gastritis_K29':
        row['has_dau_bung'] = 1
        row['has_non_oi'] = 1
    elif disease_choice == 'Diabetes_E11':
        row['GLUCOSE'] = np.random.uniform(7.8, 16.5)
    elif disease_choice == 'LiverDisease_K76':
        row['AST'] = np.random.uniform(65, 280)
        row['ALT'] = np.random.uniform(85, 340)
    
    data.append(row)
    labels.append(disease_choice)

df = pd.DataFrame(data)
le = LabelEncoder()
y = le.fit_transform(labels)
X = df[FEATURE_COLS]
print('Dữ liệu sẵn sàng:', X.shape, 'Số nhãn bệnh:', len(le.classes_))

In [ ]:
# 3. Huấn luyện XGBoost Classifier
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.08,
    objective='multi:softprob',
    random_state=42,
    tree_method='hist'
)

model.fit(X_train, y_train)
preds = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, preds))
print(classification_report(y_test, preds, target_names=le.classes_))

In [ ]:
# 4. Xuất Model và Label Encoder
model.save_model('./models_weights/tabular_xgboost.json')
joblib.dump(le, './models_weights/label_encoder.pkl')
print('Đã lưu model XGBoost vào ./models_weights/tabular_xgboost.json')